In [ ]:
using System;
using System.Collections.Generic;
using System.Linq;

namespace InvoiceSystem
{
    // Класс позиции фактуры
    public class LineItem
    {
        public string Description { get; set; } = string.Empty;
        public decimal Quantity { get; set; }
        public decimal UnitPrice { get; set; }
        public decimal Total => Quantity * UnitPrice;

        // Дополнительные поля для работы производных классов
        public DateTime? SupplyDate { get; set; }
        public string? CancellationReason { get; set; }
        public bool IsReturned { get; set; }
    }

    
    public class Invoice
    {
        public string InvoiceNumber { get; set; }
        public DateTime IssueDate { get; set; }
        public decimal TotalAmount { get; protected set; }

        protected List<LineItem> LineItems { get; set; } = new List<LineItem>();

        public Invoice(string invoiceNumber, DateTime issueDate)
        {
            InvoiceNumber = invoiceNumber;
            IssueDate = issueDate;
        }

        
        public virtual void CalculateTotal()
        {
            TotalAmount = LineItems.Sum(item => item.Total);
        }

        public virtual void AddLine(LineItem lineItem)
        {
            LineItems.Add(lineItem);
            CalculateTotal();
        }

        public virtual void RemoveLine(LineItem lineItem)
        {
            LineItems.Remove(lineItem);
            CalculateTotal();
        }

        public void PrintInfo()
        {
            Console.WriteLine($"Invoice #{InvoiceNumber} | Date: {IssueDate:dd.MM.yyyy} | Total: {TotalAmount:C}");
        }
    }

   
    public class GoodsInvoice : Invoice
    {
        public DateTime SupplyDate { get; set; }

        public GoodsInvoice(string invoiceNumber, DateTime issueDate, DateTime supplyDate)
            : base(invoiceNumber, issueDate)
        {
            SupplyDate = supplyDate;
        }

        
        public override void AddLine(LineItem lineItem)
        {
            lineItem.SupplyDate = SupplyDate;
            base.AddLine(lineItem);
            Console.WriteLine($"[GoodsInvoice] Item added: '{lineItem.Description}'. Supply date set to: {SupplyDate:dd.MM.yyyy}");
        }
    }

   
    public class ServiceInvoice : Invoice
    {
        public DateTime ServiceDate { get; set; }
        
        private readonly Dictionary<LineItem, string> _cancellationLog = new Dictionary<LineItem, string>();
        
        public string PendingCancellationReason { get; set; } = string.Empty;

        public ServiceInvoice(string invoiceNumber, DateTime issueDate, DateTime serviceDate)
            : base(invoiceNumber, issueDate)
        {
            ServiceDate = serviceDate;
        }

        public override void RemoveLine(LineItem lineItem)
        {
            if (string.IsNullOrEmpty(PendingCancellationReason))
            {
                PendingCancellationReason = "Not specified";
            }

            _cancellationLog[lineItem] = PendingCancellationReason;
            lineItem.CancellationReason = PendingCancellationReason;
            
            base.RemoveLine(lineItem);
            Console.WriteLine($"[ServiceInvoice] Service removed: '{lineItem.Description}'. Reason: {PendingCancellationReason}");
            PendingCancellationReason = string.Empty; // Сброс для следующего вызова
        }

        public void PrintCancellationLog()
        {
            Console.WriteLine("Cancellation history:");
            foreach (var log in _cancellationLog)
            {
                Console.WriteLine($"  - {log.Key.Description}: {log.Value}");
            }
        }
    }


    public class CombinedInvoice : Invoice
    {
        public bool ReturnAllowed { get; set; }

        public CombinedInvoice(string invoiceNumber, DateTime issueDate, bool returnAllowed)
            : base(invoiceNumber, issueDate)
        {
            ReturnAllowed = returnAllowed;
        }

        public override void CalculateTotal()
        {
            if (!ReturnAllowed)
            {
                base.CalculateTotal();
                Console.WriteLine("[CombinedInvoice] Returns not allowed. Standard calculation applied.");
                return;
            }

            decimal fullAmount = LineItems.Sum(item => item.Total);
            decimal returnedAmount = LineItems.Where(item => item.IsReturned).Sum(item => item.Total);
            TotalAmount = fullAmount - returnedAmount;

            Console.WriteLine($"[CombinedInvoice] Returns allowed. Adjusted total: {TotalAmount:C} (Returned: {returnedAmount:C})");
        }
    }


    class Program
    {
        static void Main()
        {
            Console.WriteLine("=== 1. GoodsInvoice ===");
            var goodsInv = new GoodsInvoice("G-1001", DateTime.Now, new DateTime(2026, 05, 10));
            goodsInv.AddLine(new LineItem { Description = "Laptop", Quantity = 2, UnitPrice = 1200m });
            goodsInv.AddLine(new LineItem { Description = "Mouse", Quantity = 5, UnitPrice = 25m });
            goodsInv.PrintInfo();

            Console.WriteLine("\n=== 2. ServiceInvoice ===");
            var serviceInv = new ServiceInvoice("S-2002", DateTime.Now, new DateTime(2026, 04, 20));
            serviceInv.AddLine(new LineItem { Description = "Consulting", Quantity = 10, UnitPrice = 150m });
            serviceInv.AddLine(new LineItem { Description = "Setup", Quantity = 1, UnitPrice = 500m });
            
            serviceInv.PendingCancellationReason = "Client request";
            serviceInv.RemoveLine(serviceInv.LineItems.Last()); 
            serviceInv.PrintInfo();
            serviceInv.PrintCancellationLog();

            Console.WriteLine("\n=== 3. CombinedInvoice ===");
            var combinedInv = new CombinedInvoice("C-3003", DateTime.Now, true);
            combinedInv.AddLine(new LineItem { Description = "Software License", Quantity = 1, UnitPrice = 3000m });
            
            var returnedItem = new LineItem { Description = "Hardware Cable", Quantity = 10, UnitPrice = 50m };
            combinedInv.AddLine(returnedItem);
            
    
            returnedItem.IsReturned = true;
            
            combinedInv.CalculateTotal(); // 
            combinedInv.PrintInfo();

            Console.WriteLine("\n=== Polymorphism Demonstration ===");
            List<Invoice> invoices = new List<Invoice> { goodsInv, serviceInv, combinedInv };
            foreach (var inv in invoices)
            {
                inv.PrintInfo(); // Вызов базового метода (полиморфизм подтипов не меняет сигнатуру, но показывает работу наследования)
            }
        }
    }
}